# CIC-IDS2018 — Dataset Overview (Google Colab)

Cloud-execution version of Notebook 07.

This notebook is designed for CIC-IDS2018 because the dataset is too large for reliable local processing.


In [8]:
from pathlib import Path
import csv
import pandas as pd
import numpy as np

DATA_DIR = Path("/content/drive/MyDrive/CIC-IDS2018")
OUTPUT_DIR = Path("/content/cicids2018_07_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_DIR)
print("Exists:", DATA_DIR.exists())


Dataset: /content/drive/MyDrive/CIC-IDS2018
Exists: False


In [9]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted:", DATA_DIR.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: False


In [ ]:
import os
import subprocess
from pathlib import Path

DATA_DIR = Path("/content/CIC-IDS2018")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Official CIC-IDS2018 dataset URL
DATASET_URL = "https://www.unb.ca/cic/datasets/ids-2018.html"

print("Dataset directory:", DATA_DIR)
print("Dataset page:", DATASET_URL)

# Check whether the dataset is already present
csv_files = list(DATA_DIR.glob("*.csv"))

if csv_files:
    print(f"\nDataset already present — found {len(csv_files)} CSV files.")
else:
    print("""
No local dataset found.

CIC-IDS2018 is very large, so downloading the complete dataset
directly into the Colab runtime may take considerable time and
storage.

For the full dataset, use the official CIC download source rather
than uploading the files manually.
""")

Dataset directory: /content/CIC-IDS2018
Dataset page: https://www.unb.ca/cic/datasets/ids-2018.html

No local dataset found.

CIC-IDS2018 is very large, so downloading the complete dataset
directly into the Colab runtime may take considerable time and
storage.

For the full dataset, use the official CIC download source rather
than uploading the files manually.



In [11]:
files = sorted(p for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".csv")
print(f"CSV files found: {len(files)}")
for f in files:
    print("-", f.name)


CSV files found: 0


## 1. File Inventory


In [12]:
file_inventory = pd.DataFrame({
    "file_name": [f.name for f in files],
    "file_size_bytes": [f.stat().st_size for f in files],
})
file_inventory["file_size_mb"] = (file_inventory["file_size_bytes"] / 1024**2).round(2)
file_inventory["file_size_gb"] = (file_inventory["file_size_bytes"] / 1024**3).round(3)
display(file_inventory)
file_inventory.to_csv(OUTPUT_DIR / "file_inventory.csv", index=False)


,file_name,file_size_bytes,file_size_mb,file_size_gb


## 2. Header-Only Schema Extraction


In [13]:
def read_csv_header(file_path):
    with open(file_path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        return next(csv.reader(f))

schemas = {}
for f in files:
    schemas[f.name] = read_csv_header(f)
    print(f"{f.name}: {len(schemas[f.name])} columns")


In [15]:
from pathlib import Path

print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("\nContents:")
    for p in list(DATA_DIR.iterdir())[:30]:
        print(p)
else:
    print("\n❌ DATA_DIR does not exist")

DATA_DIR: /content/CIC-IDS2018
Exists: True

Contents:


In [16]:
files = sorted(DATA_DIR.rglob("*.csv"))

print(f"CSV files found: {len(files)}")

for f in files[:30]:
    print(f)

CSV files found: 0


## Execution Environment

CIC-IDS2018 is substantially larger than CIC-IDS2017 and exceeded
the practical memory/runtime limits of the local development
environment.

This notebook was executed in Google Colab using the CIC-IDS2018
dataset obtained through the Kaggle dataset mirror. The dataset
itself is not included in the repository.

The analysis uses memory-efficient operations to avoid loading the
complete dataset into memory wherever possible.

In [20]:
!kaggle datasets list -s "CIC-IDS2018"

ref                                                        title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  ------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
dhoogla/csecicids2018                                      CSE-CIC-IDS2018                                     633773771  2022-08-11 22:13:03.093000           6623         46                1  
solarmainframe/ids-intrusion-csv                           IDS 2018 Intrusion CSVs (CSE-CIC-IDS2018)          1716901169  2020-10-01 23:35:58.803000          49592        239        0.9705882  
ekkykharismadhany/csecicids2018-cleaned                    cse-cic-ids2018_cleaned                             132088203  2021-10-07 13:35:08.987000           1600         31        0.9411765  
dhoogla/nfcsecicids2018v2     

In [21]:
!kaggle datasets list -s "CIC-IDS2018" --max-size 10

ref                                title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------  ------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
ruyixing/cse-cic-ids2018-improved  cse-cic-ids2018-improved                          10635644700  2026-05-15 21:15:25.587000              6          0  0.23529412       
wuliqiang/leon-nids-classil        LEON Class-IL NIDS Benchmark: Data and OFRA Code   1980073089  2026-08-11 09:50:10.463000            240          0  0.47058824       


In [23]:
from pathlib import Path
import subprocess

DATA_DIR = Path("/content/CIC-IDS2018")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading CSE-CIC-IDS2018...")

In [24]:
!kaggle datasets download \
    -d solarmainframe/ids-intrusion-csv \
    -p /content/CIC-IDS2018

Dataset URL: https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv
License(s): Attribution 4.0 International (CC BY 4.0)
100% 1.60G/1.60G [00:16<00:00, 104MB/s] 



In [25]:
!ls -lh /content/CIC-IDS2018

total 1.6G
-rw-r--r-- 1 root root 1.6G Oct  2  2020 ids-intrusion-csv.zip


In [26]:
import zipfile

archives = list(DATA_DIR.glob("*.zip"))

print("Archives:", archives)

for archive in archives:
    print(f"Extracting {archive.name}...")
    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(DATA_DIR)

print("\nExtraction complete.")

Archives: [PosixPath('/content/CIC-IDS2018/ids-intrusion-csv.zip')]
Extracting ids-intrusion-csv.zip...

Extraction complete.


In [28]:
from pathlib import Path

DATA_DIR = Path("/content/CIC-IDS2018")

print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

print("\nEverything under DATA_DIR:")
for p in DATA_DIR.rglob("*"):
    print(p)

DATA_DIR: /content/CIC-IDS2018
Exists: True

Everything under DATA_DIR:
/content/CIC-IDS2018/ids-intrusion-csv.zip
/content/CIC-IDS2018/02-14-2018.csv
/content/CIC-IDS2018/02-23-2018.csv
/content/CIC-IDS2018/02-16-2018.csv
/content/CIC-IDS2018/03-01-2018.csv
/content/CIC-IDS2018/02-20-2018.csv
/content/CIC-IDS2018/02-28-2018.csv
/content/CIC-IDS2018/02-15-2018.csv
/content/CIC-IDS2018/02-22-2018.csv
/content/CIC-IDS2018/03-02-2018.csv
/content/CIC-IDS2018/02-21-2018.csv


In [29]:
files = sorted(DATA_DIR.rglob("*.csv"))

print(f"CSV files found: {len(files)}")

for f in files:
    print(f.name)

CSV files found: 10
02-14-2018.csv
02-15-2018.csv
02-16-2018.csv
02-20-2018.csv
02-21-2018.csv
02-22-2018.csv
02-23-2018.csv
02-28-2018.csv
03-01-2018.csv
03-02-2018.csv


In [30]:
schemas = {}

for f in files:
    schemas[f.name] = read_csv_header(f)
    print(f"{f.name}: {len(schemas[f.name])} columns")

02-14-2018.csv: 80 columns
02-15-2018.csv: 80 columns
02-16-2018.csv: 80 columns
02-20-2018.csv: 84 columns
02-21-2018.csv: 80 columns
02-22-2018.csv: 80 columns
02-23-2018.csv: 80 columns
02-28-2018.csv: 80 columns
03-01-2018.csv: 80 columns
03-02-2018.csv: 80 columns


In [31]:
reference_columns = schemas[files[0].name]
column_inventory = pd.DataFrame({
    "column_index": range(len(reference_columns)),
    "column_name": reference_columns
})
display(column_inventory)
column_inventory.to_csv(OUTPUT_DIR / "column_inventory.csv", index=False)


,column_index,column_name
0,0,Dst Port
1,1,Protocol
2,2,Timestamp
3,3,Flow Duration
4,4,Tot Fwd Pkts
...,...,...
75,75,Idle Mean
76,76,Idle Std
77,77,Idle Max
78,78,Idle Min


## 3. Schema Hygiene


In [32]:
schema_hygiene = pd.DataFrame({
    "column_index": range(len(reference_columns)),
    "column_name": reference_columns
})
schema_hygiene["leading_whitespace"] = schema_hygiene["column_name"] != schema_hygiene["column_name"].str.lstrip()
schema_hygiene["trailing_whitespace"] = schema_hygiene["column_name"] != schema_hygiene["column_name"].str.rstrip()
schema_hygiene["empty_name"] = schema_hygiene["column_name"].str.strip() == ""
schema_hygiene["has_whitespace_issue"] = schema_hygiene["leading_whitespace"] | schema_hygiene["trailing_whitespace"]
display(schema_hygiene[schema_hygiene["has_whitespace_issue"] | schema_hygiene["empty_name"]])
schema_hygiene.to_csv(OUTPUT_DIR / "schema_hygiene.csv", index=False)

duplicates = pd.Series(reference_columns).value_counts()
print("Duplicate column names:")
display(duplicates[duplicates > 1])


,column_index,column_name,leading_whitespace,trailing_whitespace,empty_name,has_whitespace_issue


Duplicate column names:


,count


## 4. Schema Consistency


In [33]:
reference_set = set(reference_columns)
schema_comparison = []
for f in files:
    cols = schemas[f.name]
    current_set = set(cols)
    schema_comparison.append({
        "file_name": f.name,
        "column_count": len(cols),
        "matches_reference_order": cols == reference_columns,
        "matches_reference_set": current_set == reference_set,
        "missing_columns": len(reference_set - current_set),
        "extra_columns": len(current_set - reference_set),
    })
schema_comparison = pd.DataFrame(schema_comparison)
display(schema_comparison)
schema_comparison.to_csv(OUTPUT_DIR / "schema_comparison.csv", index=False)


,file_name,column_count,matches_reference_order,matches_reference_set,missing_columns,extra_columns
0,02-14-2018.csv,80,True,True,0,0
1,02-15-2018.csv,80,True,True,0,0
2,02-16-2018.csv,80,True,True,0,0
3,02-20-2018.csv,84,False,False,0,4
4,02-21-2018.csv,80,True,True,0,0
5,02-22-2018.csv,80,True,True,0,0
6,02-23-2018.csv,80,True,True,0,0
7,02-28-2018.csv,80,True,True,0,0
8,03-01-2018.csv,80,True,True,0,0
9,03-02-2018.csv,80,True,True,0,0


## 5. Target / Label Identification


In [34]:
label_candidates = [
    c for c in reference_columns
    if any(k in c.lower() for k in ["label", "class", "attack"])
]
print("Potential target columns:")
for c in label_candidates:
    print(repr(c))


Potential target columns:
'Label'


In [ ]:
TARGET_COLUMN = label_candidates[0]
print("Selected target:", repr(TARGET_COLUMN))


Selected target: 'Label'


In [36]:
target_presence = pd.DataFrame([
    {"file_name": f.name, "target_present": TARGET_COLUMN in schemas[f.name]}
    for f in files
])
display(target_presence)
target_presence.to_csv(OUTPUT_DIR / "target_presence.csv", index=False)
assert target_presence["target_present"].all(), "Target column missing from one or more files."


,file_name,target_present
0,02-14-2018.csv,True
1,02-15-2018.csv,True
2,02-16-2018.csv,True
3,02-20-2018.csv,True
4,02-21-2018.csv,True
5,02-22-2018.csv,True
6,02-23-2018.csv,True
7,02-28-2018.csv,True
8,03-01-2018.csv,True
9,03-02-2018.csv,True


## 6. Record Counts


In [37]:
def count_csv_rows(file_path, chunk_size=32 * 1024 * 1024):
    count = 0
    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            count += chunk.count(b"\n")
    return max(count - 1, 0)

row_counts = []
for f in files:
    rows = count_csv_rows(f)
    row_counts.append({"file_name": f.name, "record_count": rows})
    print(f"{f.name}: {rows:,} rows")

row_counts = pd.DataFrame(row_counts)
display(row_counts)
row_counts.to_csv(OUTPUT_DIR / "row_counts.csv", index=False)
total_records = int(row_counts["record_count"].sum())
print(f"Total records: {total_records:,}")


02-14-2018.csv: 1,048,575 rows
02-15-2018.csv: 1,048,575 rows
02-16-2018.csv: 1,048,575 rows
02-20-2018.csv: 7,948,748 rows
02-21-2018.csv: 1,048,575 rows
02-22-2018.csv: 1,048,575 rows
02-23-2018.csv: 1,048,575 rows
02-28-2018.csv: 613,104 rows
03-01-2018.csv: 331,125 rows
03-02-2018.csv: 1,048,575 rows


,file_name,record_count
0,02-14-2018.csv,1048575
1,02-15-2018.csv,1048575
2,02-16-2018.csv,1048575
3,02-20-2018.csv,7948748
4,02-21-2018.csv,1048575
5,02-22-2018.csv,1048575
6,02-23-2018.csv,1048575
7,02-28-2018.csv,613104
8,03-01-2018.csv,331125
9,03-02-2018.csv,1048575


Total records: 16,233,002


## 7. Traffic-Class Distribution

In [38]:
def count_target_values(file_path, target_column, chunksize=500_000):
    counts = {}
    for chunk in pd.read_csv(file_path, usecols=[target_column], chunksize=chunksize, low_memory=True):
        values = chunk[target_column].value_counts(dropna=False)
        for label, count in values.items():
            key = "<NA>" if pd.isna(label) else str(label)
            counts[key] = counts.get(key, 0) + int(count)
    return counts


In [39]:
class_results = []
for f in files:
    print(f"Processing: {f.name}")
    counts = count_target_values(f, TARGET_COLUMN)
    for label, count in counts.items():
        class_results.append({"file_name": f.name, "label": label, "record_count": count})

class_distribution = pd.DataFrame(class_results)
display(class_distribution.sort_values(["file_name", "record_count"], ascending=[True, False]))
class_distribution.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)


Processing: 02-14-2018.csv
Processing: 02-15-2018.csv
Processing: 02-16-2018.csv
Processing: 02-20-2018.csv
Processing: 02-21-2018.csv
Processing: 02-22-2018.csv
Processing: 02-23-2018.csv
Processing: 02-28-2018.csv
Processing: 03-01-2018.csv
Processing: 03-02-2018.csv


,file_name,label,record_count
2,02-14-2018.csv,Benign,667626
0,02-14-2018.csv,FTP-BruteForce,193360
1,02-14-2018.csv,SSH-Bruteforce,187589
3,02-15-2018.csv,Benign,996077
4,02-15-2018.csv,DoS attacks-GoldenEye,41508
5,02-15-2018.csv,DoS attacks-Slowloris,10990
6,02-16-2018.csv,DoS attacks-Hulk,461912
7,02-16-2018.csv,Benign,446772
8,02-16-2018.csv,DoS attacks-SlowHTTPTest,139890
9,02-16-2018.csv,Label,1


In [40]:
overall_class_distribution = (
    class_distribution.groupby("label", dropna=False)["record_count"]
    .sum().sort_values(ascending=False).reset_index()
)
overall_class_distribution["percentage"] = (
    overall_class_distribution["record_count"] /
    overall_class_distribution["record_count"].sum() * 100
)
display(overall_class_distribution)
overall_class_distribution.to_csv(OUTPUT_DIR / "overall_class_distribution.csv", index=False)


,label,record_count,percentage
0,Benign,13484708,83.069712
1,DDOS attack-HOIC,686012,4.226033
2,DDoS attacks-LOIC-HTTP,576191,3.549504
3,DoS attacks-Hulk,461912,2.845512
4,Bot,286191,1.763020
5,FTP-BruteForce,193360,1.191154
6,SSH-Bruteforce,187589,1.155603
7,Infilteration,161934,0.997560
8,DoS attacks-SlowHTTPTest,139890,0.861763
9,DoS attacks-GoldenEye,41508,0.255701


## 8. Class Presence Across Source Files


In [41]:
class_presence = (
    class_distribution.assign(present=True)
    .pivot_table(index="label", columns="file_name", values="present", fill_value=False)
)
display(class_presence)
class_presence.to_csv(OUTPUT_DIR / "class_presence_matrix.csv")


file_name,02-14-2018.csv,02-15-2018.csv,02-16-2018.csv,02-20-2018.csv,02-21-2018.csv,02-22-2018.csv,02-23-2018.csv,02-28-2018.csv,03-01-2018.csv,03-02-2018.csv
label,,,,,,,,,,
Benign,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Bot,False,False,False,False,False,False,False,False,False,1.0
Brute Force -Web,False,False,False,False,False,1.0,1.0,False,False,False
Brute Force -XSS,False,False,False,False,False,1.0,1.0,False,False,False
DDOS attack-HOIC,False,False,False,False,1.0,False,False,False,False,False
DDOS attack-LOIC-UDP,False,False,False,False,1.0,False,False,False,False,False
DDoS attacks-LOIC-HTTP,False,False,False,1.0,False,False,False,False,False,False
DoS attacks-GoldenEye,False,1.0,False,False,False,False,False,False,False,False
DoS attacks-Hulk,False,False,1.0,False,False,False,False,False,False,False


## 9. Dataset Summary


In [42]:
dataset_summary = pd.DataFrame([
    {"metric": "Source files", "value": len(files)},
    {"metric": "Total records", "value": total_records},
    {"metric": "Columns", "value": len(reference_columns)},
    {"metric": "Target column", "value": TARGET_COLUMN},
    {"metric": "Traffic classes", "value": int(overall_class_distribution["label"].nunique())},
    {"metric": "Schema consistent", "value": bool(schema_comparison["matches_reference_order"].all())},
])
display(dataset_summary)
dataset_summary.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)


,metric,value
0,Source files,10
1,Total records,16233002
2,Columns,80
3,Target column,Label
4,Traffic classes,16
5,Schema consistent,False


## 10. Conclusion

The CIC-IDS2018 dataset overview established a structural baseline using a metadata-first and streaming-based analysis approach.

The dataset consists of **10 source CSV files**, containing a total of **16,233,002 records**. The reference schema contains **80 columns**, with **`Label`** identified as the target column. All 10 source files contain the target column.

Schema inspection identified an important structural difference in `02-20-2018.csv`, which contains **84 columns** compared with the 80-column reference schema used by the other source files. This confirms that the CIC-IDS2018 source files should **not be assumed to have a perfectly uniform schema** and that the discrepancy must be investigated during the subsequent data-quality analysis.

The dataset contains multiple traffic classes distributed across the source files. The class-distribution and class-presence artifacts provide the baseline required to assess class imbalance and capture-session coverage in the next stages of analysis.

Due to the size of CIC-IDS2018, the analysis deliberately avoided loading the complete dataset into memory. File metadata and headers were processed directly, record counts were obtained using memory-efficient processing, and traffic-class distributions were calculated by processing the `Label` column incrementally. This approach allowed the complete dataset to be analyzed without requiring the entire 16.2-million-record dataset to reside in memory simultaneously.

No data cleaning, transformation, duplicate removal, class balancing, feature selection, or modelling was performed in this notebook. The purpose of this stage was strictly to establish the structural and class-distribution baseline.

The results establish that **CIC-IDS2018 is substantially larger and structurally more complex to process than CIC-IDS2017**, making memory-efficient processing an important consideration for all subsequent analysis. The identified schema discrepancy and the large dataset scale should therefore be explicitly considered in the upcoming **Feature & Data Quality Analysis** before any preprocessing or ML suitability decisions are made.

## 11. Downloadable Result Set

Generated artifacts:

```text
07_dataset_overview/
├── file_inventory.csv
├── column_inventory.csv
├── schema_hygiene.csv
├── schema_comparison.csv
├── target_presence.csv
├── row_counts.csv
├── class_distribution.csv
├── overall_class_distribution.csv
├── class_presence_matrix.csv
└── dataset_summary.csv
```

Download the generated ZIP and extract it locally to:

`results/cicids2018/07_dataset_overview/`


In [43]:
import shutil

zip_path = shutil.make_archive(
    "/content/cicids2018_07_results",
    "zip",
    OUTPUT_DIR
)
print("ZIP created:", zip_path)


ZIP created: /content/cicids2018_07_results.zip
